In [ ]:
# Copyright 2024 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# BRK3-069 - Advanced developer's guide to MLOps on Vertex AI


| | |
|-|-|
| Authors | [Elia Secchi](https://github.com/eliasecchig)

## Overview
Just like any Generative AI application, AI agents require thorough evaluation to ensure they perform reliably and effectively. This evaluation should happen both during development using test datasets (offline) and potentially in real-time (online). Developers building agent applications face a significant challenge in evaluating their performance, especially regarding the sequence and accuracy of tool usage (trajectory). Both subjective (human feedback) and objective (measurable metrics) evaluations are essential for building trust in agent behavior.

**Vertex AI Model Evaluation** provides a toolkit of quality-controlled and explainable methods and metrics to evaluate generative models and applications, including agents. It allows you to benchmark evaluation results against your own judgment using custom criteria.

This tutorial demonstrates how to build, evaluate, compare, and deploy a LangGraph agent using Vertex AI services:

*   **Build:** Construct a local agent using LangGraph capable of using tools to answer product queries.
*   **Evaluate:** Use Vertex AI Gen AI Evaluation to assess the agent's performance with a Gemini model, focusing on:
    *   Single tool usage accuracy.
    *   Tool call trajectory correctness (sequence and order).
    *   Final response quality (e.g., coherence, safety).
*   **Compare:** Repeat the evaluation using a Gemma 3 model deployed on a Vertex AI Endpoint to compare its tool-using capabilities against Gemini.
*   **Deploy:** Showcase deploying the agent using **Vertex AI Agent Engine** for scalable serving.

The tutorial uses the following Google Cloud services and resources:

*   Vertex AI Evaluation
*   Vertex AI Model Garden (Gemini, Gemma 3 via Endpoint)
*   Vertex AI Agent Engine


## Get started

### Install Vertex AI SDK and other required packages


In [ ]:
%pip install --user \
    "google-cloud-aiplatform[agent_engines,evaluation]==1.86.0" \
    "langchain==0.3.21" \
    "langchain-community==0.3.20" \
    "langchain-google-vertexai==2.0.7" \
    "langgraph==0.3.7" \
    "litellm==1.65.0"

### Restart runtime

To use the newly installed packages in this Jupyter runtime, you must restart the runtime. You can do this by running the cell below, which restarts the current kernel.

The restart might take a minute or longer. After it's restarted, continue to the next step.

In [ ]:
import IPython

app = IPython.Application.instance()
app.kernel.do_shutdown(True)

<div class="alert alert-block alert-warning">
<b>⚠️ The kernel is going to restart. In Colab or Colab Enterprise, you might see an error message that says "Your session crashed for an unknown reason." This is expected. Wait until it's finished before continuing to the next step. ⚠️</b>
</div>


### Set Google Cloud project information and initialize Vertex AI SDK

To get started using Vertex AI, you must have an existing Google Cloud project and [enable the Vertex AI API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com).

Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [ ]:
# Use the environment variable if the user doesn't provide Project ID.
import os

import vertexai

PROJECT_ID = "cloud-next-25-demo-elia"  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}

if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))

LOCATION = os.environ.get("GOOGLE_CLOUD_REGION", "us-central1")

EXPERIMENT_NAME = "evaluate-langgraph-agent"  # @param {type:"string"}

BUCKET_NAME = "[your-bucket-name]"  # @param {type: "string", placeholder: "[your-bucket-name]", isTemplate: true}

if not BUCKET_NAME or BUCKET_NAME == "[your-bucket-name]":
    BUCKET_NAME = f"{PROJECT_ID}-bucket"

BUCKET_URI = f"gs://{BUCKET_NAME}"

! gsutil mb -p $PROJECT_ID -l $LOCATION $BUCKET_URI

EXPERIMENT_NAME = "evaluate-agent-next25"  # @param {type:"string"}

vertexai.init(
    project=PROJECT_ID,
    location=LOCATION,
    staging_bucket=BUCKET_URI,
    experiment=EXPERIMENT_NAME,
)

## Import libraries

Import tutorial libraries.

In [114]:
# Standard library imports
import ast
import datetime
import re
import traceback
import uuid
from typing import Any, Callable, ClassVar, Dict, List, Literal, Optional, Sequence, Type, Union

# Third-party imports
import litellm
import pandas as pd
from IPython.display import HTML, Markdown, display
from google.auth import default
from google.auth.transport.requests import Request
from google.cloud import aiplatform

# LangChain imports
from langchain.load import dump as langchain_load_dump
from langchain_core.callbacks import CallbackManagerForLLMRun
from langchain_core.language_models.chat_models import BaseChatModel, LanguageModelInput
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage, ToolCall, ToolMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.pydantic_v1 import BaseModel
from langchain_core.runnables import Runnable, RunnableConfig
from langchain_core.tools import BaseTool, tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_google_vertexai import ChatVertexAI

# LangGraph imports
from langgraph.graph import END, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode

# Vertex AI imports
from vertexai import agent_engines
from vertexai.preview.evaluation import EvalTask
from vertexai.preview.evaluation.metrics import TrajectorySingleToolUse

## Define helper functions

Initiate a set of helper functions to print tutorial results.

In [115]:
def parse_messages_to_output_dictionary(messages) -> dict:
    """Parse response and function calls from a list of messages in the constructor format."""

    final_output = {
        "response": "No AI response found in the message history.",
        "predicted_trajectory": [],
    }

    # Handle the case where messages is a dictionary with a 'messages' key
    if isinstance(messages, dict) and "messages" in messages:
        messages = messages["messages"]
    elif isinstance(messages, str):
        # If messages is a string, we can't process it
        return final_output

    # Process each message
    function_calls = []
    for message in messages:
        # Check if it's a Tool message which contains the actual response
        if isinstance(message, dict):
            if message.get("type") == "constructor" and "ToolMessage" in message.get("id", []):
                final_output["response"] = message["kwargs"]["content"]

            # Check if it's an AI message to get tool calls
            elif message.get("type") == "constructor" and "AIMessage" in message.get("id", []):
                tool_calls = message["kwargs"].get("tool_calls", [])
                for tool_call in tool_calls:
                    if tool_call:
                        function_calls.append(
                            {
                                "tool_name": tool_call.get("name"),
                                "tool_input": tool_call.get("args"),
                            }
                        )
        # Handle LangChain message objects
        elif hasattr(message, "type") and hasattr(message, "content"):
            if message.type == "tool":
                final_output["response"] = message.content
            elif message.type == "ai" and hasattr(message, "tool_calls"):
                for tool_call in message.tool_calls:
                    if tool_call:
                        function_calls.append(
                            {
                                "tool_name": tool_call.name,
                                "tool_input": tool_call.args,
                            }
                        )

    final_output["predicted_trajectory"] = function_calls
    return final_output


def format_output_as_markdown(output: dict) -> str:
    """Convert the output dictionary to a formatted markdown string."""
    markdown = "### AI Response\n"
    markdown += f"{output['response']}\n\n"

    if output["predicted_trajectory"]:
        markdown += "### Function Calls\n"
        for call in output["predicted_trajectory"]:
            markdown += f"- **Function**: `{call['tool_name']}`\n"
            markdown += "  - **Arguments**:\n"
            for key, value in call["tool_input"].items():
                markdown += f"    - `{key}`: `{value}`\n"

    return markdown


def display_eval_report(eval_result: pd.DataFrame) -> None:
    """Display the evaluation results."""
    metrics_df = pd.DataFrame.from_dict(eval_result.summary_metrics, orient="index").T
    display(Markdown("### Summary Metrics"))
    display(metrics_df)

    display(Markdown(f"### Row-wise Metrics"))
    display(eval_result.metrics_table)


def display_drilldown(row: pd.Series) -> None:
    """Displays a drill-down view for trajectory data within a row."""

    style = "white-space: pre-wrap; width: 800px; overflow-x: auto;"

    if not (
        isinstance(row["predicted_trajectory"], list)
        and isinstance(row["reference_trajectory"], list)
    ):
        return

    for predicted_trajectory, reference_trajectory in zip(
        row["predicted_trajectory"], row["reference_trajectory"]
    ):
        display(
            HTML(
                f"<h3>Tool Names:</h3><div style='{style}'>{predicted_trajectory['tool_name'], reference_trajectory['tool_name']}</div>"
            )
        )

        if not (
            isinstance(predicted_trajectory.get("tool_input"), dict)
            and isinstance(reference_trajectory.get("tool_input"), dict)
        ):
            continue

        for tool_input_key in predicted_trajectory["tool_input"]:
            print("Tool Input Key: ", tool_input_key)

            if tool_input_key in reference_trajectory["tool_input"]:
                print(
                    "Tool Values: ",
                    predicted_trajectory["tool_input"][tool_input_key],
                    reference_trajectory["tool_input"][tool_input_key],
                )
            else:
                print(
                    "Tool Values: ",
                    predicted_trajectory["tool_input"][tool_input_key],
                    "N/A",
                )
        print("\n")
    display(HTML("<hr>"))


def display_dataframe_rows(
    df: pd.DataFrame,
    columns: list[str] | None = None,
    num_rows: int = 3,
    display_drilldown: bool = False,
) -> None:
    """Displays a subset of rows from a DataFrame, optionally including a drill-down view."""

    if columns:
        df = df[columns]

    base_style = "font-family: monospace; font-size: 14px; white-space: pre-wrap; width: auto; overflow-x: auto;"
    header_style = base_style + "font-weight: bold;"

    for _, row in df.head(num_rows).iterrows():
        for column in df.columns:
            display(
                HTML(
                    f"<span style='{header_style}'>{column.replace('_', ' ').title()}: </span>"
                )
            )
            display(HTML(f"<span style='{base_style}'>{row[column]}</span><br>"))

        display(HTML("<hr>"))

        if (
            display_drilldown
            and "predicted_trajectory" in df.columns
            and "reference_trajectory" in df.columns
        ):
            display_drilldown(row)


In [116]:
def create_gemma_tool_calling_model():
    """Returns a custom ChatGemma class that supports tool calling functionality."""

    class ChatGemmaWithToolCalls(BaseChatModel):
        """
        A chat model implementation for Gemma that supports tool calling functionality.

        This class wraps the Gemma model through LiteLLM and provides support for function/tool
        calling by parsing the model's responses for tool invocations wrapped in code blocks.
        """
        model: str
        api_base: str
        api_key: str
        temperature: Optional[float] = None
        max_tokens: Optional[int] = None
        timeout: Optional[int] = None
        model_kwargs: Dict[str, Any] = {}
        _bound_tools_openai_fmt: Optional[List[Dict[str, Any]]] = None

        _tool_system_prompt_template: ClassVar[str] = """\
    At each turn, if you decide to invoke any of the function(s), it should be wrapped with ```tool_code```.
    The python methods described below are imported and available, you can only use defined methods. The generated code should be readable and efficient. The response to a method will be wrapped in ```tool_output``` use it to call more tools or generate a helpful, friendly response.

    The following Python methods are available:
    {rendered_tools}
    """

        @property
        def _llm_type(self) -> str:
            return "gemma-chat-model-litellm-ast"

        @property
        def _identifying_params(self) -> Dict[str, Any]:
            return {
                "model": self.model,
                "api_base": self.api_base,
                "temperature": self.temperature,
                "max_tokens": self.max_tokens,
                "timeout": self.timeout,
                "model_kwargs": self.model_kwargs,
            }

        def _get_litellm_params(self, stop: Optional[List[str]], **kwargs: Any) -> Dict[str, Any]:
            params = {
                "model": self.model,
                "api_base": self.api_base,
                "api_key": self.api_key,
                **self.model_kwargs,
                **{k: v for k, v in {
                    "temperature": self.temperature,
                    "max_tokens": self.max_tokens,
                    "timeout": self.timeout,
                    "stop": stop,
                }.items() if v is not None},
                **kwargs,
            }
            return params

        def _prepare_messages_and_system_prompt(
            self, messages: List[BaseMessage], tools: Optional[List[Dict[str, Any]]]
        ) -> List[Dict[str, Any]]:
            litellm_messages = []
            system_content_parts = []

            for msg in messages:
                if isinstance(msg, SystemMessage):
                    system_content_parts.append(str(msg.content))
                elif isinstance(msg, HumanMessage):
                    litellm_messages.append({"role": "user", "content": str(msg.content)})
                elif isinstance(msg, AIMessage):
                    message_content = str(msg.content)
                    if hasattr(msg, 'tool_calls') and msg.tool_calls:
                        tool_codes = []
                        for tool_call in msg.tool_calls:
                            args_str = ', '.join([f'{k}=\"{v}\"' for k, v in tool_call['args'].items()])
                            tool_codes.append(f"```tool_code\n{tool_call['name']}({args_str})\n```")

                        if message_content:
                            message_content = f"{message_content}\n\n{''.join(tool_codes)}"
                        else:
                            message_content = ''.join(tool_codes)
                    litellm_messages.append({"role": "assistant", "content": message_content})
                elif isinstance(msg, ToolMessage):
                    litellm_messages.append({"role": "user", "content": f"Tool '{msg.name}' returned: {str(msg.content)}"})

            final_system_content = "\n\n".join(system_content_parts) if system_content_parts else None

            if tools:
                try:
                    rendered_tools = _render_tools_as_python_signatures(tools)
                    tool_instructions = self._tool_system_prompt_template.format(rendered_tools=rendered_tools)
                    final_system_content = f"{final_system_content}\n\n{tool_instructions}" if final_system_content else tool_instructions
                except Exception as e:
                    print(f"--- Warning: Failed to render tools for prompt injection: {e} ---")

            if final_system_content:
                litellm_messages.insert(0, {"role": "system", "content": final_system_content})

            return litellm_messages

        def _parse_tool_code_block_ast(self, raw_content: str, tools: Optional[List[Dict[str, Any]]]) -> Optional[List[Dict[str, Any]]]:
            if not tools:
                return None

            block_pattern = r"```tool_code\s*(.*?)\s*```"
            match = re.search(block_pattern, raw_content, re.DOTALL | re.IGNORECASE)
            if not match:
                return None

            code_to_parse = match.group(1).strip()
            if not code_to_parse:
                return None

            try:
                tree = ast.parse(code_to_parse)
                known_tool_names = {t['function']['name'] for t in tools if 'function' in t and 'name' in t['function']}
                if not known_tool_names:
                    return None

                visitor = ToolCallVisitor(known_tool_names)
                visitor.visit(tree)

                return [visitor.found_tool_call] if visitor.found_tool_call else None

            except SyntaxError:
                print(f"--- Warning: Invalid Python syntax in tool_code block ---")
                return None
            except Exception as e:
                print(f"--- Warning: Error during AST parsing or visiting: {e} ---")
                return None

        def _generate(
            self,
            messages: List[BaseMessage],
            stop: Optional[List[str]] = None,
            run_manager: Optional[CallbackManagerForLLMRun] = None,
            **kwargs: Any,
        ) -> ChatResult:
            current_tools = self._bound_tools_openai_fmt
            if "tools" in kwargs and isinstance(kwargs["tools"], list):
                current_tools = kwargs.pop("tools")

            litellm_messages = self._prepare_messages_and_system_prompt(messages, current_tools)
            params = self._get_litellm_params(stop, **kwargs)
            params["messages"] = litellm_messages

            try:
                response = litellm.completion(**params)
                choice = response.choices[0]
                raw_content = choice.message.content or ""

                parsed_tool_calls = self._parse_tool_code_block_ast(raw_content, current_tools)

                if parsed_tool_calls:
                    tool_calls = [
                        ToolCall(name=tc["name"], args=tc["args"], id=tc["id"])
                        for tc in parsed_tool_calls if tc
                    ]
                    ai_message = AIMessage(content="", tool_calls=tool_calls)
                else:
                    ai_message = AIMessage(
                        content=raw_content,
                        response_metadata={
                            "model": response.model,
                            "usage": response.usage.model_dump() if hasattr(response, 'usage') and response.usage else {},
                            "finish_reason": getattr(choice, 'finish_reason', None),
                        },
                    )

                return ChatResult(generations=[ChatGeneration(message=ai_message)])

            except Exception as e:
                print(f"LLM call or processing failed: {str(e)}")
                traceback.print_exc()
                return ChatResult(generations=[ChatGeneration(message=AIMessage(content=f"LLM call failed: {str(e)}"))])


        def bind_tools(
            self,
            tools: Sequence[Union[Dict[str, Any], Type[BaseModel], Callable, BaseTool]],
            **kwargs: Any,
        ) -> Runnable[LanguageModelInput, BaseMessage]:
            formatted_tools = [convert_to_openai_tool(tool) for tool in tools]
            self._bound_tools_openai_fmt = formatted_tools
            return super().bind(tools=formatted_tools, **kwargs)


    def _render_tools_as_python_signatures(tools: List[Dict[str, Any]]) -> str:
        signatures = []
        for tool in tools:
            func = tool.get('function', {})
            name = func.get('name', 'unknown_tool')
            description = func.get('description', '')
            params = func.get('parameters', {}).get('properties', {})
            param_defs = []
            arg_details = []
            type_map = {'string': 'str', 'number': 'float', 'integer': 'int', 'boolean': 'bool', 'array': 'list', 'object': 'dict'}
            for p_name, p_info in params.items():
                py_type = type_map.get(p_info.get('type', 'any'), 'Any')
                param_defs.append(f"{p_name}: {py_type}")
                arg_details.append(f"  {p_name} ({py_type}): {p_info.get('description', '')}")
            param_str = ", ".join(param_defs)
            args_doc = "\n".join(arg_details)
            signature = f"def {name}({param_str}) -> Any:\n    \"\"\"{description}\n\n    Args:\n{args_doc}\n    \"\"\""
            signatures.append(signature)
        return f"```python\n{chr(10)}{chr(10).join(signatures)}\n```"

    class ToolCallVisitor(ast.NodeVisitor):
        def __init__(self, known_tool_names: set):
            self.known_tool_names = known_tool_names
            self.found_tool_call: Optional[Dict[str, Any]] = None

        def visit_Call(self, node: ast.Call):
            if self.found_tool_call:
                return

            if isinstance(node.func, ast.Name) and node.func.id in self.known_tool_names:
                tool_name = node.func.id
                args_dict = {}
                try:
                    for i, arg_node in enumerate(node.args):
                        try: args_dict[f"arg_{i}"] = ast.literal_eval(arg_node)
                        except ValueError: args_dict[f"arg_{i}"] = f"<AST Node: {type(arg_node).__name__}>"

                    for keyword in node.keywords:
                        if keyword.arg:
                            try: args_dict[keyword.arg] = ast.literal_eval(keyword.value)
                            except ValueError: args_dict[keyword.arg] = f"<AST Node: {type(keyword.value).__name__}>"

                    self.found_tool_call = {
                        "name": tool_name,
                        "args": args_dict,
                        "id": f"call_{tool_name}_{uuid.uuid4()}"
                    }
                except Exception as e:
                    print(f"--- Error processing arguments for {tool_name} in AST: {e} ---")
                    self.found_tool_call = None

            if not self.found_tool_call:
                self.generic_visit(node)
    return ChatGemmaWithToolCalls

ChatGemmaWithToolCalls = create_gemma_tool_calling_model()

## Build LangGraph agent

Build your application using LangGraph, including the Gemini model, custom tools that you define and a router to control the conversational flow.

### Set tools

To start, set the tools that a customer support agent needs to do their job.

In [117]:
@tool
def get_product_details(product_name: str) -> str:
    """Retrieves detailed product information including features, specifications, and benefits for a given product name. Returns a descriptive string about the product or a message if the product is not found."""
    details = {
        "wavephone": "A cutting-edge smartphone with advanced camera features and lightning-fast processing.",
        "powercharge ultra": "A super fast and light usb charger with multiple ports and quick-charge technology.",
        "cloudrunner x200": "High-performance running shoes designed for comfort, support, and speed.",
        "soundwave pro": "Wireless headphones with advanced noise cancellation technology for immersive audio.",
        "homebuddy mini": "A voice-controlled smart speaker that plays music, sets alarms, and controls smart home devices.",
    }
    return details.get(product_name.lower(), "Product details not found.")


@tool
def get_product_price(product_name: str) -> int:
    """Retrieves the current price of a product in USD. Returns an integer price value for the specified product or an error message if the product is not found."""
    details = {
        "wavephone": 500,
        "powercharge ultra": 10,
        "cloudrunner x200": 100,
        "soundwave pro": 50,
        "homebuddy mini": 80,
    }
    return details.get(product_name.lower(), "Product price not found.")

### Assemble the agent

The Vertex AI Gen AI Evaluation works directly with 'Queryable' agents, and also lets you add your own custom functions with a specific structure (signature).

In this case, you assemble the agent using a custom function. The function triggers the agent for a given input and parse the agent outcome to extract the response and called tools.

In [118]:
class LangGraphApp:
    def __init__(
        self,
        project: str,
        location: str,
        model: Optional[str] = None,
        endpoint_id: Optional[str] = None,
    ) -> None:
        if not (model or endpoint_id):
            raise ValueError("Either 'model' or 'endpoint_id' must be provided.")
        if model and endpoint_id:
            raise ValueError("Provide either 'model' or 'endpoint_id', not both.")

        self.project_id = project
        self.location = location
        self.model_name = model
        self.endpoint_id = endpoint_id

    # The set_up method is used to define application initialization logic
    def set_up(self) -> None:
        if self.model_name:
            model = ChatVertexAI(model=self.model_name, project=self.project_id, location=self.location)
        elif self.endpoint_id:
            # Use ChatLiteLLM for endpoint_id
            creds, _ = default()
            creds.refresh(Request())

            model = ChatGemmaWithToolCalls(
                model="openai/",
                api_base=(
                    f"https://{self.endpoint_id}.{self.location}-fasttryout.prediction.vertexai.goog/"
                    f"v1/projects/{self.project_id}/locations/{self.location}/endpoints/{self.endpoint_id}/"
                ),
                api_key=creds.token,
                temperature=0
            )

        else:
            # This case should not be reached due to __init__ validation
            raise ValueError("No model or endpoint_id configured.")

        tools = [get_product_details, get_product_price]
        model_with_tools = model.bind_tools(tools)

        def should_continue(state: MessagesState) -> str:
            """Determines whether to use tools or end the conversation."""
            last_message = state["messages"][-1]
            return "tools" if last_message.tool_calls else END


        def call_model(state: MessagesState, config: RunnableConfig) -> dict[str, BaseMessage]:
            """Calls the language model and returns the response."""
            system_message = "You are a helpful AI assistant."
            messages_with_system = [{"type": "system", "content": system_message}] + state[
                "messages"
            ]
            response = model_with_tools.invoke(messages_with_system, config)
            return {"messages": response}

        workflow = StateGraph(MessagesState)
        workflow.add_node("agent", call_model)
        workflow.add_node("tools", ToolNode(tools))
        workflow.set_entry_point("agent")

        # 5. Define graph edges
        workflow.add_conditional_edges("agent", should_continue)
        workflow.add_edge("tools", "agent")

        # 6. Compile the workflow
        self.app = workflow.compile()

    # The query method will be used to send inputs to the agent
    def query(self, input: str):
        """Query the application."""
        response = langchain_load_dump.dumpd(self.app.invoke({"messages":[HumanMessage(input)]}))
        return response

agent = LangGraphApp(project=PROJECT_ID, location=LOCATION, model="gemini-2.0-flash-001")
agent.set_up()

In [ ]:
from IPython.display import Image, display

display(Image(agent.app.get_graph().draw_mermaid_png()))

### Test the agent

Query your agent.

In [ ]:
response = agent.query("Get product details for wavephone")
display(Markdown(format_output_as_markdown(parse_messages_to_output_dictionary(response))))

In [ ]:
response = agent.query("Get product price for homebuddy mini")
display(Markdown(format_output_as_markdown(parse_messages_to_output_dictionary(response))))

## Evaluating a LangGraph agent with Vertex AI Gen AI Evaluation

When working with AI agents, it's important to keep track of their performance and how well they're working. You can look at this in two main ways: **monitoring** and **observability**.

Monitoring focuses on how well your agent is performing specific tasks:

* **Single Tool Selection**: Is the agent choosing the right tools for the job?

* **Multiple Tool Selection (or Trajectory)**: Is the agent making logical choices in the order it uses tools?

* **Response generation**: Is the agent's output good, and does it make sense based on the tools it used?

Observability is about understanding the overall health of the agent:

* **Latency**: How long does it take the agent to respond?

* **Failure Rate**: How often does the agent fail to produce a response?

Vertex AI Gen AI Evaluation service helps you to assess all of these aspects both while you are prototyping the agent or after you deploy it in production. It provides [pre-built evaluation criteria and metrics](https://cloud.google.com/vertex-ai/generative-ai/docs/models/determine-eval) so you can see exactly how your agents are doing and identify areas for improvement.

### Prepare Agent Evaluation dataset

To evaluate your AI agent using the Vertex AI Gen AI Evaluation service, you need a specific dataset depending on what aspects you want to evaluate of your agent.  

This dataset should include the prompts given to the agent. It can also contain the ideal or expected response (ground truth) and the intended sequence of tool calls the agent should take (reference trajectory) representing the sequence of tools you expect agent calls for each given prompt.

> Optionally, you can provide both generated responses and predicted trajectory (**Bring-Your-Own-Dataset scenario**).

Below you have an example of dataset you might have with a customer support agent with user prompt and the reference trajectory.

In [122]:
eval_data = {
    "prompt": [
        "Get price for wavephone",
        "Get product details and price for soundwave pro",
        "Get details for powercharge ultra",
        "Get product details and price for cloudrunner x200",
        "Get product details for homebuddy mini",
    ],
    "reference_trajectory": [
        [
            {
                "tool_name": "get_product_price",
                "tool_input": {"product_name": "wavephone"},
            }
        ],
        [
            {
                "tool_name": "get_product_details",
                "tool_input": {"product_name": "soundwave pro"},
            },
            {
                "tool_name": "get_product_price",
                "tool_input": {"product_name": "soundwave pro"},
            },
        ],
        [
            {
                "tool_name": "get_product_details",
                "tool_input": {"product_name": "powercharge ultra"},
            }
        ],
        [
            {
                "tool_name": "get_product_details",
                "tool_input": {"product_name": "cloudrunner x200"},
            },
            {
                "tool_name": "get_product_price",
                "tool_input": {"product_name": "cloudrunner x200"}
            },
        ],
        [
            {
                "tool_name": "get_product_details",
                "tool_input": {"product_name": "homebuddy mini"},
            }
        ],
    ],
}

eval_sample_dataset = pd.DataFrame(eval_data)

Print some samples from the dataset.

In [ ]:
display_dataframe_rows(eval_sample_dataset, num_rows=3)

### Define Evaluation Metrics

We define several types of metrics to evaluate different aspects of the agent's performance:

*   **Single Tool Usage:** Checks if a *specific* tool was used correctly (using `TrajectorySingleToolUse`).
*   **Trajectory Metrics:** Evaluate the entire sequence of tool calls against the reference trajectory (e.g., exact match, order).
*   **Response Metrics:** Assess the quality of the final generated text response (e.g., safety, coherence).

In [ ]:
single_tool_usage_metrics = [TrajectorySingleToolUse(tool_name="get_product_price")]

In [124]:
trajectory_metrics = [
    "trajectory_exact_match",
    "trajectory_in_order_match",
    "trajectory_any_order_match",
    "trajectory_precision",
    "trajectory_recall",
]


In [ ]:
response_metrics = ["safety", "coherence"]

In [125]:
# Combine all metrics into a single list for evaluation
eval_metrics = single_tool_usage_metrics + trajectory_metrics + response_metrics

#### Run an evaluation task

Run a new agent's evaluation.

In [ ]:
def runnable_eval(input):
    response = agent.query(input)
    return parse_messages_to_output_dictionary(response)

In [ ]:
EXPERIMENT_RUN = f"eval-gemini-2-flash-{datetime.datetime.now().strftime('%Y%m%d%H%M%S')}"

response_eval_tool_task = EvalTask(
    dataset=eval_sample_dataset,
    metrics=eval_metrics,
    experiment=EXPERIMENT_NAME,
)

response_eval_tool_result = response_eval_tool_task.evaluate(
    runnable=runnable_eval, experiment_run_name=EXPERIMENT_RUN
)

display_eval_report(response_eval_tool_result)

## Let's compare it with Gemma3!


We are ready to test the Gemma 3 endpoint we just created via Model Garden!

> Note: The evaluation metrics used (`safety`, `coherence`, tool usage metrics, trajectory metrics) are specific to this task and the prompts defined earlier.

In [127]:
GEMMA3_ENDPOINT_ID = "2413766672549675008"
GEMMA3_LOCATION = "us-west1"

In [128]:
# Generate an Auth token
creds, _ = default()
creds.refresh(Request())

In [129]:
# We can start interacting with Gemma3

gemma3_llm = ChatGemmaWithToolCalls(
    model="openai/",
    api_base=(
        f"https://{GEMMA3_ENDPOINT_ID}.{GEMMA3_LOCATION}-fasttryout.prediction.vertexai.goog/"
        f"v1/projects/{PROJECT_ID}/locations/{GEMMA3_LOCATION}/endpoints/{GEMMA3_ENDPOINT_ID}/"
    ),
    api_key=creds.token,
    temperature=0
)

In [ ]:
gemma3_llm.invoke("hi")

In [131]:
agent = LangGraphApp(project=PROJECT_ID, location=GEMMA3_LOCATION, endpoint_id=GEMMA3_ENDPOINT_ID)
agent.set_up()

In [ ]:
response = agent.query("Get product price of homebuddy mini")

display(Markdown(format_output_as_markdown(parse_messages_to_output_dictionary(response))))

In [ ]:
EXPERIMENT_RUN = f"eval-gemma-3-flash-{datetime.datetime.now().strftime('%Y%m%d%H%M%S')}"

response_eval_tool_task = EvalTask(
    dataset=eval_sample_dataset,
    metrics=eval_metrics,
    experiment=EXPERIMENT_NAME,
)

response_eval_tool_result = response_eval_tool_task.evaluate(
    runnable=runnable_eval, experiment_run_name=EXPERIMENT_RUN
)

display_eval_report(response_eval_tool_result)

## Agent Development
### Deploying with Agent Engine

Vertex AI Agent Engine (formerly known as LangChain on Vertex AI or Vertex AI Reasoning Engine) is a fully managed Google Cloud service enabling developers to deploy, manage, and scale AI agents in production. Agent Engine handles the infrastructure to scale agents in production so you can focus on creating intelligent and impactful applications. Vertex AI Agent Engine offers:

- **Fully managed**: Deploy and scale agents with a managed runtime that provides robust security features including VPC-SC compliance and comprehensive end-to-end management capabilities. Gain CRUD access to multi-agent applications that use Google Cloud Trace (supporting OpenTelemetry) for performance monitoring and tracing.

- **Quality and evaluation**: Ensure agent quality with the integrated Gen AI Evaluation service.

- **Simplified development**: Vertex AI Agent Engine abstracts away low-level tasks such as application server development and configuration of authentication and IAM, allowing you to focus on the unique capabilities of your agent, such as its behavior, tools, and model parameters.

- **Framework agnostic**: Enjoy flexibility when deploying agents that you build using different python frameworks including LangGraph, Langchain, AG2, and CrewAI. If you already have an existing agent, you can adapt it to run on Vertex AI Agent Engine using the custom template in the SDK.




![Agent Engine Architecture](https://cloud.google.com/static/vertex-ai/generative-ai/docs/agent-engine/images/agent-engine.png)


In [ ]:
agent = LangGraphApp(project=PROJECT_ID, location=LOCATION, model="gemini-2.0-flash-001")

In [ ]:
remote_agent = agent_engines.create(
    agent,
    requirements=[
        "cloudpickle==3.0.0",
        "google-cloud-aiplatform[evaluation]>=1.84.0",
        "google-genai>=1.2.0",
        "langchain>=0.3.21",
        "langchain-community>=0.3.20",
        "langchain-google-vertexai>=2.0.7",
        "langgraph>=0.3.7",
        "litellm>=1.65.0"
    ],
)

### Deploying via the Agent Starter Pack 🚀

The Agent Starter Pack provides a production-ready template for building and deploying agents in Google Cloud.

![Agent Starter Pack Banner](https://github.com/GoogleCloudPlatform/agent-starter-pack/blob/main/docs/images/ags_banner.png?raw=true)

In [134]:
REMOTE_AGENT_ID = 'projects/669402927687/locations/us-central1/reasoningEngines/2553690522300448768'
remote_agent = agent_engines.get(REMOTE_AGENT_ID)

In [ ]:
response = remote_agent.query(input={"messages": [{"type": "human", "content": "What's the price of the homebuddy mini"}]})
display(
    Markdown(format_output_as_markdown(parse_messages_to_output_dictionary(response)))
)

In [ ]:
response_stream = remote_agent.stream_query(input={"messages": [{"type": "human", "content": "What's the price of the homebuddy mini"}]})
for chunk in response_stream:
    print(chunk)

## Cleaning up


In [ ]:
delete_experiment = True

if delete_experiment:
    try:
        experiment = aiplatform.Experiment(EXPERIMENT_NAME)
        experiment.delete(delete_backing_tensorboard_runs=True)
    except Exception as e:
        print(e)